In [1]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("test.csv")
df = pd.read_csv(DATA_PATH)

print(f"Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas")
df.head()

Dimensiones: 3,263 filas x 4 columnas


,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


# Laboratorio 5: análisis de tweets

Este laboratorio documenta la descripción de los datos, el preprocesamiento, el análisis de unigramas y bigramas, y un modelo preliminar de clasificación.

> **Alcance:** `test.csv` no contiene una columna objetivo (`target`). Por ello, el modelo usa una etiqueta proxy transparente: `keyword` presente (`1`) frente a `keyword` ausente (`0`). Esta aproximación sirve para validar el pipeline, pero no sustituye una etiqueta de desastre real.

## 1. Descripción de los datos

El archivo contiene tweets identificados por `id`, una palabra clave (`keyword`), una ubicación (`location`) y el texto (`text`). La ubicación puede ser nula y la palabra clave también. Como el texto es la fuente principal para el análisis lingüístico, se conserva aunque falten metadatos.

In [2]:
# Diagnóstico inicial: tipos, valores faltantes y duplicados.
profile = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "porcentaje_nulos": (df.isna().mean() * 100).round(2),
    "valores_unicos": df.nunique(),
})
print("Duplicados completos:", int(df.duplicated().sum()))
profile

Duplicados completos: 0


,tipo,nulos,porcentaje_nulos,valores_unicos
id,int64,0,0.00,3263
keyword,object,26,0.80,221
location,object,1105,33.86,1602
text,object,0,0.00,3243


**Interpretación:** `id` no tiene faltantes y permite identificar cada observación. `text` tampoco tiene faltantes ni textos vacíos, así que no es necesario eliminar filas por ausencia de contenido. `location` tiene muchos valores faltantes y se tratará como metadato opcional; no se usará como predictor en este primer modelo para evitar introducir ruido.

## 2. Preprocesamiento

Se normaliza el texto para análisis léxico: se convierte a minúsculas, se eliminan URLs, menciones, HTML, signos y números, y se conservan únicamente tokens alfabéticos de longitud mínima 2. No se elimina `text` original. Las palabras vacías se excluyen de unigramas y bigramas porque aportan poca información semántica. Para el modelo, el vectorizador aplicará su propia representación TF-IDF.

In [3]:
import html
import re

STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "has",
    "have", "he", "her", "his", "i", "in", "is", "it", "its", "me", "my",
    "of", "on", "or", "our", "that", "the", "their", "this", "to", "was",
    "we", "were", "with", "you", "your",
}


def clean_text(text: str) -> str:
    text = html.unescape(str(text)).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+|#\w+", " ", text)
    tokens = re.findall(r"[a-záéíóúüñ]{2,}", text)
    return " ".join(token for token in tokens if token not in STOPWORDS)


df_processed = df.copy()
df_processed["text_clean"] = df_processed["text"].map(clean_text)
df_processed["keyword_present"] = df_processed["keyword"].notna().astype(int)

print("Textos vacíos después de limpiar:", int(df_processed["text_clean"].eq("").sum()))
df_processed[["text", "text_clean", "keyword_present"]].head()

Textos vacíos después de limpiar: 0


,text,text_clean,keyword_present
0,Just happened a terrible car crash,just happened terrible car crash,0
1,"Heard about #earthquake is different cities, s...",heard about different cities stay safe everyone,0
2,"there is a forest fire at spot pond, geese are...",there forest fire spot pond geese fleeing acro...,0
3,Apocalypse lighting. #Spokane #wildfires,apocalypse lighting,0
4,Typhoon Soudelor kills 28 in China and Taiwan,typhoon soudelor kills china taiwan,0


## 3. Unigramas y bigramas

Los unigramas son tokens individuales y muestran los términos más frecuentes. Los bigramas son pares consecutivos y ayudan a observar expresiones recurrentes. Se calculan sobre `text_clean`, por lo que no cuentan URLs, menciones, hashtags ni palabras vacías.

In [4]:
from collections import Counter

unigram_counter = Counter()
bigram_counter = Counter()
for text in df_processed["text_clean"]:
    tokens = text.split()
    unigram_counter.update(tokens)
    bigram_counter.update(zip(tokens, tokens[1:]))

unigrams = pd.DataFrame(unigram_counter.most_common(20), columns=["unigrama", "frecuencia"])
bigrams = pd.DataFrame(
    [(" ".join(pair), count) for pair, count in bigram_counter.most_common(20)],
    columns=["bigrama", "frecuencia"],
)

print("Top 20 unigramas")
display(unigrams)
print("Top 20 bigramas")
display(bigrams)

Top 20 unigramas


,unigrama,frecuencia
0,like,145
1,up,141
2,out,140
3,just,135
4,no,135
5,not,132
6,will,114
7,all,113
8,but,113
9,so,113


Top 20 bigramas


,bigrama,frecuencia
0,suicide bomber,32
1,first responders,22
2,more than,20
3,liked video,19
4,emergency services,18
5,year old,17
6,full read,17
7,suicide bombing,17
8,burning buildings,17
9,northern california,16


## 4. Modelo preliminar de clasificación

Se construye un modelo binario para predecir `keyword_present`. La representación TF-IDF usa unigramas y bigramas, y la regresión logística incorpora `class_weight="balanced"` para compensar el desbalance. La división 80/20 es estratificada y se fija `random_state` para reproducibilidad. Se reportan precisión, recall y F1 por clase, además de exactitud.

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df_processed["text_clean"],
    df_processed["keyword_present"],
    test_size=0.20,
    random_state=42,
    stratify=df_processed["keyword_present"],
)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.98, sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

print(f"Registros de entrenamiento: {len(X_train):,}")
print(f"Registros de prueba: {len(X_test):,}")
print(f"Características TF-IDF: {X_train_tfidf.shape[1]:,}")
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred, target_names=["sin_keyword", "con_keyword"], zero_division=0))
print("Matriz de confusión (filas=real, columnas=predicha):")
print(confusion_matrix(y_test, y_pred))

Registros de entrenamiento: 2,610
Registros de prueba: 653
Características TF-IDF: 4,671

Reporte de clasificación:
              precision    recall  f1-score   support

 sin_keyword       0.14      0.20      0.17         5
 con_keyword       0.99      0.99      0.99       648

    accuracy                           0.98       653
   macro avg       0.57      0.60      0.58       653
weighted avg       0.99      0.98      0.99       653

Matriz de confusión (filas=real, columnas=predicha):
[[  1   4]
 [  6 642]]


## 5. Hallazgos y conclusiones

- El conjunto tiene `3,263` tweets y `3,243` textos distintos; no se encontraron duplicados completos.
- `text` está completo. `keyword` tiene 26 valores faltantes (0.80%) y `location` 1,105 (33.86%), por lo que la ubicación se conserva como información descriptiva, pero no se usa en el modelo preliminar.
- La limpieza produjo cero textos vacíos. El análisis léxico encontró vocabulario asociado a situaciones de emergencia, y los bigramas aportaron más contexto que los unigramas aislados.
- El modelo preliminar alcanzó accuracy de 0.98, pero el F1 macro fue 0.58 y la clase `sin_keyword` obtuvo F1 de 0.17. Esto muestra que el desbalance domina la accuracy.
- **Limitación principal:** `keyword_present` es una etiqueta proxy construida a partir de un campo del propio CSV, no una etiqueta de relevancia o de desastre. El modelo solo valida el pipeline técnico; no permite concluir todavía que detecte tweets sobre desastres.

Para una siguiente versión se necesita un archivo con `target` real (por ejemplo, `1` si el tweet describe un desastre y `0` si no), o una anotación manual balanceada. Con esa etiqueta se deben repetir la división estratificada, las métricas por clase y, preferiblemente, validación cruzada.